# Alpha-Mind Quick Start 8: IC Decay Calculation

> This example calculates IC decay for the EMA5D factor.

In [ ]:
import asyncio
import datetime as dt
import numpy as np


## 1. Basic Parameter Settings

In [ ]:
# Set factor names, centered factors, and related settings
import pyarrow as pa
import pyarrow.compute as pc

rng = np.random.default_rng(42)


* Shift the factor with the `SHIFT` formula function.

In [ ]:
n_dates = 6
n_assets = 12
dates = [dt.datetime(2026, 1, 5) + dt.timedelta(days=index) for index in range(n_dates)]


* Fetch the underlying data.

In [ ]:
codes = np.array([f"{index:06d}" for index in range(1, n_assets + 1)])
n_rows = n_dates * n_assets


* Neutralize the factor.

In [ ]:
panel = pa.table({
    "trade_date": [value for value in dates for _ in range(n_assets)],
    "code": np.tile(codes, n_dates),
    "factor": rng.normal(size=n_rows),
    "value": rng.normal(size=n_rows),
    "dx": rng.normal(0.0, 0.01, size=n_rows),
    "industry": np.tile(np.array(["Bank", "Auto", "Health"]), n_dates * n_assets // 3),
    "benchmark_weight": np.full(n_rows, 1.0 / n_assets),
    "BETA": np.tile(np.linspace(0.8, 1.2, n_assets) ** 2, n_dates),
    "SIZE": np.tile(np.linspace(-1.0, 1.0, n_assets), n_dates),
})
factor = panel["factor"].to_numpy(zero_copy_only=False).reshape(n_dates, n_assets)
returns = panel["dx"].to_numpy(zero_copy_only=False).reshape(n_dates, n_assets)


* Calculate the IC series.

In [ ]:
records = []
for horizon in range(3):
    for start in range(n_dates - horizon):
        ic = np.corrcoef(factor[start], returns[start + horizon])[0, 1]
        records.append(pa.table({
            "horizon": [horizon], "trade_date": [dates[start]], "ic": [ic]
        }))
decay = pa.concat_tables(records).group_by("horizon").aggregate([("ic", "mean")])


* Draw a bar chart.

In [ ]:
decay = decay.sort_by("horizon")
assert decay["horizon"].to_pylist() == [0, 1, 2]
decay
